# 10. Chạy riêng một mô hình × một kịch bản: xem kết quả, chỉ test, hoặc huấn luyện từ đầu tới test

**Cách dùng:** sửa ô **0. Tham số** (mô hình, kịch bản, bài toán) rồi chọn **Run All**, hoặc chạy từng mục:

| Mục | Việc | Thời gian (RTX 5090) |
|---|---|---|
| 1 | Xem kết quả đã lưu của mô hình, mọi kịch bản (đọc `results/ket_qua_thuc_nghiem.csv`) | tức thì |
| 2 | **Chỉ test:** nạp weight tốt nhất đã lưu (E2, t+1), dự báo lại tập kiểm tra | vài giây |
| 3 | **Huấn luyện từ đầu tới test:** CV 3 fold → huấn luyện cuối 2019–2024 → chấm tập kiểm tra, kèm hình | 5–20 s (LSTM, GRU) |
| 4 | (tuỳ chọn) So sánh nhiều kịch bản của mô hình chính | 1–3 phút |

- Mọi bước dùng **đúng** quy trình của kết quả chính (`src/pipeline.py`): cùng dữ liệu đóng băng v1, cùng siêu tham số, cùng hạt giống. Trên cùng GPU, kết quả trùng với kết quả chính; mục 3 tự đối chiếu và báo "trùng".
- Notebook này **không ghi file nào**: kết quả hiện ngay trong notebook. Kết quả chính sinh bằng notebook 03–06 và 09.
- Lõi ở `src/run_single.py`. Bản dòng lệnh tương đương: `experiments/chay_mo_hinh.py`.

**Các kịch bản (`EXP`):**

| Mã | Nội dung | Mô hình |
|---|---|---|
| `PERS` | Q(t+1) = Q(t) | chỉ Persistence |
| `E1` | không có dữ liệu Hủa Na | 6 mô hình |
| `E2` | + Q ra Hủa Na (**cấu hình chính**) | 6 mô hình |
| `E3` | + Q đến Hủa Na | 6 mô hình |
| `E4` | + Q ra, Q đến, mực nước Hủa Na | 6 mô hình |
| `P_F4`, `P_F5`, `P_EMA` | thực nghiệm phụ | 6 mô hình |
| `E5` | E2 + hàm mất mát có trọng số theo lưu lượng | 6 mô hình |
| `E6`, `E56` | E2 + phạt dự báo thiếu (± trọng số) | RNN, LSTM, GRU |
| `FUT` | E2 + mưa thực đo t+1 (cận trên) | 6 mô hình |
| `KH_TD` | E2 + Q ra Hủa Na t+1 thực đo (cận trên) | 6 mô hình, chỉ H = 1 |
| `KH_AD10` … `KH_AD50` | mô hình KH_TD, dự báo bằng kế hoạch xả sai số 10–50 % | 6 mô hình, chỉ H = 1 |
| `KH_SS10` … `KH_SS50` | huấn luyện và dự báo bằng kế hoạch xả sai số 10–50 % | 6 mô hình, chỉ H = 1 |

In [ ]:
# ── Google Colab: gắn Google Drive, vào thư mục dự án, cài đúng phiên bản thư viện. Chạy trên máy thì tự bỏ qua. ──
import os, sys, glob, subprocess
DU_AN = '/content/drive/MyDrive/T_DAThS_Ninh'   # ← thư mục dự án trên Google Drive (notebook 01, mục 0 tạo ra)
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir(os.path.join(DU_AN, 'src')):          # không thấy → tự tìm trong My Drive (sâu tối đa 3 cấp)
        hit = [h for k in range(1, 4) for h in glob.glob('/content/drive/MyDrive/' + '*/' * k + 'src/results_store.py')]
        assert hit, ('Chưa có dự án trên Google Drive: mở notebook 01_PhanTich_XuLy_DuLieu và chạy 2 ô của mục 0 '
                     '(lấy code từ GitHub, giải nén data/kết quả), hoặc sửa biến DU_AN ở đầu ô này')
        DU_AN = os.path.dirname(os.path.dirname(hit[0]))
    os.chdir(DU_AN)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements_colab.txt'], check=True)

import json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src' / 'results_store.py').exists():
    assert ROOT != ROOT.parent, 'Không tìm thấy thư mục dự án (thư mục chứa src/): mở notebook từ trong dự án'
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3})
from src.utils import load_config, results_dir, dataset_meta
cfg = load_config()
print('Thư mục dự án:', ROOT, '| kết quả lưu ở:', cfg['paths']['results'])
print('Thiết bị:', f'GPU {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else
      'CPU (trên Colab: Runtime → Change runtime type → GPU để chạy nhanh hơn)')
from src import run_single as R
from src.training.train import DISPLAY
from src.evaluation.metrics import per_horizon
pd.set_option('display.max_rows', 200)

## 0. Tham số

In [ ]:
MODEL  = 'lstm'   # 'persistence', 'ridge', 'extra_trees', 'xgboost', 'rnn', 'lstm', 'gru'
EXP    = 'E2'     # mã kịch bản, xem bảng ở đầu notebook
H      = 1        # 1: bài toán chính t+1; 10: bài toán mở rộng t+1 … t+10
TU_DAU = False    # True: dò lại siêu tham số (và γ, λ) bằng CV thay vì đọc config.json — khoảng 2 phút với LSTM
print(f'{DISPLAY[MODEL]} · {EXP}: {R.describe(EXP)} · H = {H}')

## 1. Kết quả đã lưu (không tính lại)

NSE CV là tiêu chí lựa chọn (trung bình 3 fold). Các cột còn lại là trung bình các hạt giống trên tập kiểm tra
01/2025 – 08/2026. ĐLC NSE: độ lệch chuẩn giữa các hạt giống.

In [ ]:
display(R.saved_results(cfg, MODEL, H=H).round(3))

## 2. Chỉ test: nạp mô hình cuối đã lưu và dự báo lại tập kiểm tra

Weight tốt nhất được lưu cho **E2, t+1** (5 hạt giống mỗi mô hình; Ridge-ARX tất định nên 1). Chạy trên CPU, nên dự
báo của RNN, LSTM, GRU có thể lệch dưới 1 m³/s so với bản đã lưu (tạo trên GPU); chỉ tiêu trùng tới 3 chữ số. Kịch bản
khác: xem mục 1 hoặc huấn luyện lại ở mục 3.

In [ ]:
try:
    t_saved, pred_saved = R.test_saved(cfg, MODEL, EXP, H)
    display(t_saved[R.SHOW + ['chênh so với dự báo đã lưu (m³/s)']].round(3))
    print('TB:', ' · '.join(f'{k} {t_saved[k].mean():.3f}' for k in ['NSE', 'KGE', 'RMSE', 'NSE_Q95']))
except FileNotFoundError as e:
    print(e)

## 3. Huấn luyện từ đầu tới test

1. CV 3 fold (huấn luyện 2019–2020 / 2019–2021 / 2019–2023; xác thực 2021 / 2022 / 2024), dừng sớm theo tập xác thực.
2. Số epoch (số cây với XGBoost) = trung vị số vòng tốt nhất trên CV.
3. Huấn luyện cuối trên 2019–2024 với 5 hạt giống, dự báo tập kiểm tra 01/2025 – 08/2026.

In [ ]:
r = R.train_one(cfg, MODEL, EXP, H, TU_DAU)

### 3.1 Kiểm định chéo theo fold (trung bình các hạt giống)

In [ ]:
display(r['cv'].groupby('fold')[['NSE', 'KGE', 'RMSE', 'MAE', 'PBIAS', 'NSE_Q95', 'best_iter']].mean().round(3))
print(f"NSE CV = {r['cv_tom_tat']['NSE CV']:.3f} (ĐLC giữa fold {r['cv_tom_tat']['NSE CV (ĐLC giữa fold)']:.3f})")

### 3.2 Tập kiểm tra theo hạt giống

In [ ]:
t = r['test'].set_index('seed')[R.SHOW + ['n_iter']]
display(pd.concat([t, t.agg(['mean', 'std'])]).round(3))
if r['so_voi_ket_qua_chinh'] is not None:
    print(f"Đối chiếu với kết quả chính: chênh NSE lớn nhất {r['so_voi_ket_qua_chinh']:.1e}")

### 3.3 Hình: dự báo t+1 trên tập kiểm tra (trung bình các hạt giống)

In [ ]:
o, p = R.mean_series(r['pred'], h=1)
fig = plt.figure(figsize=(15, 8)); gs = fig.add_gridspec(2, 3)
ax = fig.add_subplot(gs[0, :]); ax.plot(o, 'k', lw=1, label='quan trắc'); ax.plot(p, 'C3', lw=1, label=f'{DISPLAY[MODEL]} {EXP}')
ax.set_ylabel('Q đến Cửa Đạt (m³/s)'); ax.set_title('(a) Toàn bộ tập kiểm tra', loc='left'); ax.legend()
ax = fig.add_subplot(gs[1, :2]); a_, b_ = '2025-07-01', '2025-10-31'
ax.plot(o[a_:b_], 'k', lw=1.3, label='quan trắc'); ax.plot(p[a_:b_], 'C3', lw=1.2, label='dự báo')
ax.set_title('(b) Mùa lũ 2025', loc='left'); ax.legend()
ax = fig.add_subplot(gs[1, 2]); m_ = o.notna()
ax.scatter(o[m_], p[m_], s=6, alpha=.5); lim = [max(1, o.min()), o.max() * 1.1]
ax.plot(lim, lim, 'k--', lw=.8); ax.set_xscale('log'); ax.set_yscale('log'); ax.set_xlabel('quan trắc'); ax.set_ylabel('dự báo')
ax.set_title('(c) Dự báo – quan trắc (thang log)', loc='left')
plt.tight_layout(); plt.show()
if H > 1:                                           # bài toán t+1 … t+10: chỉ tiêu theo bước dự báo
    rows = []
    for s, d in r['pred'].groupby('seed'):
        w = lambda c: d.pivot(index='ngay_goc', columns='h', values=c)
        rows.append(per_horizon(w('Q_do').values, w('Q_du_bao').values, w('dich_that').astype(bool).values)[['NSE', 'MAE']])
    ph = pd.concat(rows).groupby(level=0).mean(); display(ph.round(3).T)
    ph.plot(subplots=True, marker='o', figsize=(10, 3.5), layout=(1, 2), title=['NSE theo h', 'MAE theo h']); plt.show()

## 4. (Tuỳ chọn) So sánh nhiều kịch bản của mô hình chính

Đặt `CHAY_LAI = True` để huấn luyện lại từng kịch bản (1–3 phút cho 2 mô hình × 8 kịch bản). Mặc định chỉ đọc kết quả đã
lưu.

In [ ]:
MO_HINH  = ['lstm', 'gru']
KICH_BAN = ['E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'FUT', 'KH_TD']
CHAY_LAI = False

if CHAY_LAI:
    rows = []
    for m in MO_HINH:
        for e in KICH_BAN:
            x = R.train_one(cfg, m, e, 1, log=lambda s: None)
            rows.append({'mô hình': DISPLAY[m], 'kịch bản': e, 'NSE CV': x['cv_tom_tat']['NSE CV'],
                         'NSE test': x['test'].NSE.mean(), 'ĐLC NSE test': x['test'].NSE.std(ddof=1),
                         'NSE Q>Q95 test': x['test'].NSE_Q95.mean(), 'Peak Bias (%)': x['test'].Peak_Bias.mean(),
                         'chênh so với kết quả chính': x['so_voi_ket_qua_chinh']})
            print(f'{DISPLAY[m]} {e}: xong')
    display(pd.DataFrame(rows).set_index(['mô hình', 'kịch bản']).round(3))
else:
    t = pd.concat([R.saved_results(cfg, m) for m in MO_HINH])
    display(t[t.index.get_level_values('kịch bản').isin(KICH_BAN)].round(3))